In [2]:
# 1. Upgrade pip just in case
!pip install --upgrade pip

# 2. Install a stable version of Rasterio that has pre-built binaries
# This bypasses the "GDAL/gdal-config" error by using a "wheel" file instead of compiling source code.
!pip install rasterio==1.3.10

# 3. Now install Sedona (It will see rasterio is already there and skip the bad step)
!pip install apache-sedona==1.6.1

# 4. Java is likely fine ("Nothing to do" means it's already installed), but we run it to be safe
!sudo yum install -y java-1.8.0-openjdk-devel

Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
https://download.docker.com/linux/centos/2/x86_64/stable/repodata/repomd.xml: [Errno 14] HTTPS Error 404 - Not Found
Trying other mirror.
63 packages excluded due to repository priority protections
Package 1:java-1.8.0-openjdk-devel-1.8.0.472.b08-1.amzn2.0.1.x86_64 already installed and latest version
Nothing to do


In [3]:
import time
from pyspark.sql.functions import split, explode, col, desc, broadcast
from project_setup import get_spark_session, CRIME_DATA, MO_CODES_DATA

spark = get_spark_session("Query3_Analysis")
sc = spark.sparkContext

Configuring Environment for 'Query3_Analysis'...
   Resource Config: 4 Executors | 1 Cores | 2g RAM
   JAVA_HOME set to: /usr/lib/jvm/java-1.8.0-openjdk-1.8.0.472.b08-1.amzn2.0.1.x86_64/jre
:: loading settings :: url = jar:file:/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.4_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-113ae48e-b1be-4b73-97f2-f9deda0c2505;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 in central
	found org.datasyslab#geotools-wrapper;1.6.1-28.2 in central
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 470ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 from c

25/12/15 15:01:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/12/15 15:01:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/15 15:02:01 ERROR Inbox: Ignoring error
java.lang.NullPointerException
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:579)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:121)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:103)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:100)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$spark$rpc$netty$MessageLoop$$receiveLoop(MessageLoop.scala:75)
	at org.apache.spark.rpc.netty.MessageLoop$$anon$1.run(MessageLoop.scala:41)
	at java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:511)
	at java.util.concurrent.FutureTask.run(FutureTask.java:266)
	at 

[Stage 0:>                                                          (0 + 1) / 1]

25/12/15 15:02:17 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 15:02:17 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 15:02:17 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 15:02:17 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 15:02:17 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 15:02:17 WARN SimpleFunctionRegistry: The function st_intersection_aggr replaced a previously registered function.
25/12/15 15:02:17 WARN SimpleFunctionRegistry: The function st_union_aggr replaced a previously registered function.
   Sedona Context Active


In [8]:
import time
from pyspark.sql.functions import col, split, explode, desc, broadcast

print(" Starting Query 3 (DataFrame API)...")

print("\n--- PART 1: End-to-End Benchmark (Standard Approach) ---")
spark.catalog.clearCache() 
start_time_total = time.time()

df_mo_codes = spark.read.text(MO_CODES_DATA) \
    .withColumn("split", split(col("value"), " ", 2)) \
    .select(col("split").getItem(0).alias("MO_Code"), col("split").getItem(1).alias("MO_Desc"))

df_crime = spark.read.option("header", "true").csv(CRIME_DATA)

df_exploded = df_crime.filter(col("Mocodes").isNotNull()) \
                      .withColumn("MO_Code_Crime", explode(split(col("Mocodes"), " "))) \
                      .groupBy("MO_Code_Crime").count()

joined = df_exploded.join(df_mo_codes, df_exploded["MO_Code_Crime"] == df_mo_codes["MO_Code"])

result = joined.orderBy(desc("count")).limit(5).collect()

end_time_total = time.time()
clean_df_time = end_time_total - start_time_total

print(f"Top 5 MO Codes:")
for r in result: 
    print(f"   {r['MO_Code']} - {r['MO_Desc']}: {r['count']}")
print(f"  Fair DataFrame Execution Time: {clean_df_time:.2f} seconds")


print("\n--- PART 2: Join Strategy Analysis (Cached) ---")

df_exploded.cache()
df_exploded.count() # Force cache

strategies = ["NO_HINT", "BROADCAST", "MERGE", "SHUFFLE_HASH", "SHUFFLE_REPLICATE_NL"]

print(f"\n{'Strategy':<25} | {'Time (s)':<10} | {'Actual Plan Used'}")
print("-" * 80)

for strategy in strategies:
    loop_start = time.time()
    
    if strategy == "NO_HINT":
        joined_strat = df_exploded.join(df_mo_codes, df_exploded["MO_Code_Crime"] == df_mo_codes["MO_Code"])
    elif strategy == "BROADCAST":
        joined_strat = df_exploded.join(broadcast(df_mo_codes), df_exploded["MO_Code_Crime"] == df_mo_codes["MO_Code"])
    else:
        joined_strat = df_exploded.join(df_mo_codes.hint(strategy), df_exploded["MO_Code_Crime"] == df_mo_codes["MO_Code"])
        
    joined_strat.orderBy(desc("count")).limit(5).collect()
    
    duration = time.time() - loop_start
    
    plan_string = joined_strat._jdf.queryExecution().executedPlan().toString()
    if "BroadcastHashJoin" in plan_string: plan_type = "BroadcastHashJoin"
    elif "SortMergeJoin" in plan_string: plan_type = "SortMergeJoin"
    elif "ShuffledHashJoin" in plan_string: plan_type = "ShuffledHashJoin"
    elif "CartesianProduct" in plan_string: plan_type = "CartesianProduct"
    elif "BroadcastNestedLoopJoin" in plan_string: plan_type = "BroadcastNestedLoopJoin"
    else: plan_type = "Other/Unknown"

    print(f"{strategy:<25} | {duration:<10.4f} | {plan_type}")

print("-" * 80)
df_exploded.unpersist()

 Starting Query 3 (DataFrame API)...

--- PART 1: End-to-End Benchmark (Standard Approach) ---


Top 5 MO Codes:
   0344 - Removes vict property: 1002900
   1822 - Stranger: 548422
   0416 - Hit-Hit w/ weapon: 404773
   0329 - Vandalized: 377536
   0913 - Victim knew Suspect: 278618
  Fair DataFrame Execution Time: 13.95 seconds

--- PART 2: Join Strategy Analysis (Cached) ---
25/12/15 15:16:48 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 15:16:48 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 15:16:48 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 15:16:48 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 15:16:48 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 15:16:48 WARN SimpleFunctionRegistry: The function st_intersection_aggr 


Strategy                  | Time (s)   | Actual Plan Used
--------------------------------------------------------------------------------


NO_HINT                   | 0.7848     | BroadcastHashJoin
BROADCAST                 | 0.7010     | BroadcastHashJoin


MERGE                     | 1.1274     | SortMergeJoin


SHUFFLE_HASH              | 0.8448     | ShuffledHashJoin


[Stage 109:==================================================>  (190 + 2) / 200]

SHUFFLE_REPLICATE_NL      | 4.5843     | CartesianProduct
--------------------------------------------------------------------------------


DataFrame[MO_Code_Crime: string, count: bigint]

In [6]:
# --- RDD API IMPLEMENTATION (Robust & Clean) ---
import csv
import io

print(" Starting Query 3 (RDD API)...")
start_time = time.time()

mo_rdd = sc.textFile(MO_CODES_DATA) \
           .map(lambda x: x.split(" ", 1)) \
           .filter(lambda x: len(x) == 2)

crime_rdd = sc.textFile(",".join(CRIME_DATA))
header_line = crime_rdd.first()

try:
    reader = csv.reader(io.StringIO(header_line))
    header_cols = next(reader)
    clean_cols = [c.replace('"', '').strip() for c in header_cols]
    mo_idx = clean_cols.index("Mocodes")
    print(f"    Found 'Mocodes' at index {mo_idx}")
except ValueError:
    print("     'Mocodes' column not found in header. Using default index 16.")
    mo_idx = 16

def parse_crime_line(line):
    if line == header_line: 
        return []
    
    try:
        reader = csv.reader(io.StringIO(line))
        parts = next(reader)
        
        if len(parts) <= mo_idx: 
            return []
        
        codes_str = parts[mo_idx].strip()
        if not codes_str:
            return []
            
        return [(code, 1) for code in codes_str.split(" ") if code]
        
    except Exception:
        return []

# map -> reduceByKey -> join -> sortBy -> take
result_rdd = crime_rdd.flatMap(parse_crime_line) \
                      .reduceByKey(lambda a, b: a + b) \
                      .join(mo_rdd) \
                      .sortBy(lambda x: x[1][0], ascending=False) \
                      .take(5)

print("-" * 80)
print(f"{'MO Code':<10} | {'Description':<40} | {'Count':<10}")
print("-" * 80)

for (code, (count, description)) in result_rdd:
    print(f"{code:<10} | {description:<40} | {count:<10}")

print("-" * 80)
print(f"RDD Execution Time: {time.time() - start_time:.2f} seconds")

 Starting Query 3 (RDD API)...


   ℹ️  Found 'Mocodes' at index 10


[Stage 56:====================================================>   (28 + 2) / 30]

--------------------------------------------------------------------------------
MO Code    | Description                              | Count     
--------------------------------------------------------------------------------
0344       | Removes vict property                    | 1002900   
1822       | Stranger                                 | 548422    
0416       | Hit-Hit w/ weapon                        | 404773    
0329       | Vandalized                               | 377536    
0913       | Victim knew Suspect                      | 278618    
--------------------------------------------------------------------------------
RDD Execution Time: 47.63 seconds
